# 05 - Vector Stores y Retrieval

## Curso de LLMs y Aplicaciones de IA

**Duración estimada:** 2-2.5 horas

---

## Índice

1. [Introducción a Vector Stores](#intro)
2. [FAISS: Vector Store local](#faiss)
3. [Document Loaders](#loaders)
4. [Text Splitters](#splitters)
5. [Similarity Search](#search)
6. [Persistencia y carga](#persistencia)
7. [Ejercicios prácticos](#ejercicios)

---

## Objetivos de aprendizaje

Al finalizar este notebook, serás capaz de:
- Entender qué son los vector stores y cómo funcionan
- Crear y usar FAISS para búsqueda vectorial
- Cargar documentos desde diferentes fuentes
- Dividir documentos en chunks óptimos
- Realizar búsquedas semánticas eficientes

<a name="intro"></a>
## 1. Introducción a Vector Stores

### ¿Qué es un Vector Store?

Un **Vector Store** (base de datos vectorial) es un sistema diseñado para almacenar, indexar y buscar vectores de alta dimensionalidad de forma eficiente.

### ¿Por qué son importantes?

Los LLMs tienen conocimiento limitado a su fecha de entrenamiento y no conocen datos privados/corporativos. Los vector stores permiten:

1. **Búsqueda semántica**: Encontrar documentos por significado, no keywords
2. **Memoria a largo plazo**: Almacenar conocimiento externo
3. **RAG**: Retrieval-Augmented Generation

### Opciones populares

| Vector Store | Tipo | Características |
|--------------|------|----------------|
| **FAISS** | Local | Gratuito, rápido, en memoria |
| **Chroma** | Local | Gratuito, persistente, fácil de usar |
| **Pinecone** | Cloud | Escalable, managed, pago |
| **Weaviate** | Cloud/Local | Open source, GraphQL |
| **Qdrant** | Cloud/Local | Open source, Rust |

Para este curso usamos **FAISS** por ser gratuito y no requerir infraestructura.

In [1]:
# Install required libraries (all free)
#%pip install -q langchain langchain-community langchain-huggingface
#%pip install -q faiss-cpu sentence-transformers
#%pip install -q beautifulsoup4 pypdf

In [2]:
import warnings
warnings.filterwarnings('ignore')

print("Librerías instaladas ✓")

Librerías instaladas ✓


<a name="faiss"></a>
## 2. FAISS: Vector Store local

**FAISS** (Facebook AI Similarity Search) es una librería desarrollada por Meta para búsqueda eficiente de vectores similares.

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
#from langchain.docstore.document import Document
from langchain_core.documents import Document
# Load free embedding model
print("Cargando modelo de embeddings (gratuito)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Test embedding
test_vector = embeddings.embed_query("Hello world")
print(f"Dimensión del embedding: {len(test_vector)}")

Cargando modelo de embeddings (gratuito)...


2026-05-25 17:33:03.164116: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Dimensión del embedding: 384


In [4]:
# Create simple documents
documents = [
    Document(
        page_content="Python es un lenguaje de programación interpretado y de alto nivel.",
        metadata={"source": "python.txt", "topic": "programming"}
    ),
    Document(
        page_content="JavaScript es el lenguaje de la web, ejecutado en navegadores.",
        metadata={"source": "javascript.txt", "topic": "programming"}
    ),
    Document(
        page_content="Machine Learning es un subcampo de la inteligencia artificial.",
        metadata={"source": "ml.txt", "topic": "AI"}
    ),
    Document(
        page_content="Deep Learning usa redes neuronales con múltiples capas.",
        metadata={"source": "dl.txt", "topic": "AI"}
    ),
    Document(
        page_content="FAISS es una librería para búsqueda eficiente de vectores similares.",
        metadata={"source": "faiss.txt", "topic": "tools"}
    ),
]

print(f"Creados {len(documents)} documentos")

Creados 5 documentos


In [5]:
# Create FAISS vector store
print("Creando vector store con FAISS...")
vector_store = FAISS.from_documents(documents, embeddings)

print(f"Vector store creado con {vector_store.index.ntotal} vectores")

Creando vector store con FAISS...
Vector store creado con 5 vectores


<a name="loaders"></a>
## 3. Document Loaders

LangChain proporciona loaders para cargar documentos desde múltiples fuentes.

In [6]:
# Web page loader
from langchain_community.document_loaders import WebBaseLoader

# Load content from a web page
print("Cargando contenido web...")
loader = WebBaseLoader("https://es.wikipedia.org/wiki/Python")
web_docs = loader.load()

print(f"Documentos cargados: {len(web_docs)}")
print(f"Caracteres: {len(web_docs[0].page_content)}")
print(f"Metadata: {web_docs[0].metadata}")
print(f"\nPrimeros 500 caracteres:\n{web_docs[0].page_content[:500]}...")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Cargando contenido web...
Documentos cargados: 1
Caracteres: 56052
Metadata: {'source': 'https://es.wikipedia.org/wiki/Python', 'title': 'Python - Wikipedia, la enciclopedia libre', 'language': 'es'}

Primeros 500 caracteres:




Python - Wikipedia, la enciclopedia libre
































Ir al contenido







Menú principal





Menú principal
mover a la barra lateral
ocultar



		Navegación
	


PortadaPortal de la comunidadActualidadCambios recientesPáginas nuevasPágina aleatoriaAyudaNotificar un errorPáginas especiales



















Buscar











Buscar






















Apariencia
















Donaciones

Crear una cuenta

Acceder








Herramientas personales






Donaciones


C...


In [7]:
# Text file loader (create a sample file first)
sample_text = """Introducción a la Inteligencia Artificial

La inteligencia artificial (IA) es una rama de la informática que busca crear 
sistemas capaces de realizar tareas que normalmente requieren inteligencia humana.

Tipos de IA:
1. IA Débil (Narrow AI): Diseñada para tareas específicas
2. IA Fuerte (General AI): Capaz de cualquier tarea intelectual humana
3. Superinteligencia: Hipotética IA que supera la inteligencia humana

Aplicaciones comunes:
- Reconocimiento de voz
- Visión por computadora
- Procesamiento de lenguaje natural
- Sistemas de recomendación
"""

# Save to file
with open("sample_ai_doc.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

# Load from file
from langchain_community.document_loaders import TextLoader

text_loader = TextLoader("sample_ai_doc.txt", encoding="utf-8")
text_docs = text_loader.load()

print(f"Documento cargado: {len(text_docs)} archivo(s)")
print(f"Contenido:\n{text_docs[0].page_content}")

Documento cargado: 1 archivo(s)
Contenido:
Introducción a la Inteligencia Artificial

La inteligencia artificial (IA) es una rama de la informática que busca crear 
sistemas capaces de realizar tareas que normalmente requieren inteligencia humana.

Tipos de IA:
1. IA Débil (Narrow AI): Diseñada para tareas específicas
2. IA Fuerte (General AI): Capaz de cualquier tarea intelectual humana
3. Superinteligencia: Hipotética IA que supera la inteligencia humana

Aplicaciones comunes:
- Reconocimiento de voz
- Visión por computadora
- Procesamiento de lenguaje natural
- Sistemas de recomendación



<a name="splitters"></a>
## 4. Text Splitters

Los documentos largos deben dividirse en **chunks** (fragmentos) para:
- Respetar límites de contexto del LLM
- Mejorar precisión de búsqueda
- Reducir costos de procesamiento

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,        # Maximum characters per chunk
    chunk_overlap=50,      # Overlap between chunks
    length_function=len,
    separators=["\n\n", "\n", " ", ""]  # Priority of separators
)

# Split the document
chunks = text_splitter.split_documents(text_docs)

print(f"Documento original dividido en {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} ({len(chunk.page_content)} chars):")
    print(f"  '{chunk.page_content[:80]}...'")
    print()

Documento original dividido en 5 chunks

Chunk 1 (41 chars):
  'Introducción a la Inteligencia Artificial...'

Chunk 2 (161 chars):
  'La inteligencia artificial (IA) es una rama de la informática que busca crear 
s...'

Chunk 3 (141 chars):
  'Tipos de IA:
1. IA Débil (Narrow AI): Diseñada para tareas específicas
2. IA Fue...'

Chunk 4 (69 chars):
  '3. Superinteligencia: Hipotética IA que supera la inteligencia humana...'

Chunk 5 (134 chars):
  'Aplicaciones comunes:
- Reconocimiento de voz
- Visión por computadora
- Procesa...'



In [9]:
# Different chunk sizes comparison
sizes = [100, 200, 500]

long_text = web_docs[0].page_content[:2000]  # First 2000 chars from Wikipedia

print("Comparación de tamaños de chunk:")
print("=" * 50)

for size in sizes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=size // 5  # 20% overlap
    )
    chunks = splitter.split_text(long_text)
    print(f"chunk_size={size}: {len(chunks)} chunks")

Comparación de tamaños de chunk:
chunk_size=100: 26 chunks
chunk_size=200: 13 chunks
chunk_size=500: 5 chunks


### Estrategia de chunking

| Chunk Size | Ventajas | Desventajas |
|------------|----------|-------------|
| Pequeño (100-300) | Más preciso, específico | Puede perder contexto |
| Mediano (300-800) | Balance contexto/precisión | Uso general |
| Grande (800-2000) | Más contexto | Menos preciso, más tokens |

<a name="search"></a>
## 5. Similarity Search

La búsqueda por similitud es la operación fundamental de los vector stores.

In [10]:
# Create vector store with more documents
all_chunks = text_splitter.split_documents(text_docs)

# Add more sample documents
extra_docs = [
    Document(page_content="Los transformers revolucionaron el NLP en 2017.", 
             metadata={"topic": "NLP"}),
    Document(page_content="GPT-4 es un modelo de lenguaje desarrollado por OpenAI.",
             metadata={"topic": "LLM"}),
    Document(page_content="BERT es un modelo bidireccional pre-entrenado.",
             metadata={"topic": "NLP"}),
    Document(page_content="Las redes convolucionales son excelentes para imágenes.",
             metadata={"topic": "CV"}),
    Document(page_content="El aprendizaje por refuerzo entrena agentes mediante recompensas.",
             metadata={"topic": "RL"}),
]

all_docs = all_chunks + extra_docs

# Create new vector store
vs = FAISS.from_documents(all_docs, embeddings)
print(f"Vector store con {vs.index.ntotal} documentos")

Vector store con 10 documentos


In [11]:
# Basic similarity search
query = "¿Qué es el procesamiento de lenguaje natural?"

results = vs.similarity_search(query, k=3)

print(f"Query: '{query}'")
print("\nResultados:")
print("=" * 50)
for i, doc in enumerate(results, 1):
    print(f"\n{i}. {doc.page_content}")
    print(f"   Metadata: {doc.metadata}")

Query: '¿Qué es el procesamiento de lenguaje natural?'

Resultados:

1. BERT es un modelo bidireccional pre-entrenado.
   Metadata: {'topic': 'NLP'}

2. GPT-4 es un modelo de lenguaje desarrollado por OpenAI.
   Metadata: {'topic': 'LLM'}

3. Las redes convolucionales son excelentes para imágenes.
   Metadata: {'topic': 'CV'}


In [12]:
# Similarity search with scores
results_with_scores = vs.similarity_search_with_score(query, k=3)

print(f"Query: '{query}'")
print("\nResultados con puntuación (menor = más similar):")
print("=" * 50)
for doc, score in results_with_scores:
    print(f"\nScore: {score:.4f}")
    print(f"  {doc.page_content}")

Query: '¿Qué es el procesamiento de lenguaje natural?'

Resultados con puntuación (menor = más similar):

Score: 1.1154
  BERT es un modelo bidireccional pre-entrenado.

Score: 1.1231
  GPT-4 es un modelo de lenguaje desarrollado por OpenAI.

Score: 1.1309
  Las redes convolucionales son excelentes para imágenes.


In [13]:
# Search with metadata filter
# Note: FAISS doesn't support native filtering, but we can post-filter

def search_with_filter(vector_store, query, k=10, filter_key=None, filter_value=None):
    """Search with optional metadata filtering."""
    # Get more results to filter
    results = vector_store.similarity_search(query, k=k)
    
    if filter_key and filter_value:
        results = [doc for doc in results 
                   if doc.metadata.get(filter_key) == filter_value]
    
    return results

# Search only NLP documents
nlp_results = search_with_filter(vs, "modelos de lenguaje", filter_key="topic", filter_value="NLP")

print("Búsqueda filtrada (topic='NLP'):")
for doc in nlp_results[:3]:
    print(f"  - {doc.page_content}")

Búsqueda filtrada (topic='NLP'):
  - BERT es un modelo bidireccional pre-entrenado.
  - Los transformers revolucionaron el NLP en 2017.


### Retriever: Interfaz para búsqueda

LangChain proporciona una interfaz `Retriever` para usar vector stores en chains.

In [14]:
# Create retriever from vector store
retriever = vs.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Use retriever
docs = retriever.invoke("inteligencia artificial")

print(f"Retriever devolvió {len(docs)} documentos:")
for doc in docs:
    print(f"  - {doc.page_content[:60]}...")

Retriever devolvió 3 documentos:
  - La inteligencia artificial (IA) es una rama de la informátic...
  - Introducción a la Inteligencia Artificial...
  - 3. Superinteligencia: Hipotética IA que supera la inteligenc...


<a name="persistencia"></a>
## 6. Persistencia y carga

FAISS permite guardar y cargar el índice para no tener que recalcular embeddings.

In [15]:
# Save vector store to disk
vs.save_local("faiss_index")
print("Vector store guardado en 'faiss_index/'")

Vector store guardado en 'faiss_index/'


In [16]:
# Load vector store from disk
loaded_vs = FAISS.load_local(
    "faiss_index", 
    embeddings,
    allow_dangerous_deserialization=True  # Required for pickle files
)

print(f"Vector store cargado con {loaded_vs.index.ntotal} vectores")

# Test that it works
test_results = loaded_vs.similarity_search("IA", k=1)
print(f"Test búsqueda: {test_results[0].page_content[:50]}...")

Vector store cargado con 10 vectores
Test búsqueda: Tipos de IA:
1. IA Débil (Narrow AI): Diseñada par...


In [17]:
# Add more documents to existing vector store
new_docs = [
    Document(page_content="LangChain facilita el desarrollo de aplicaciones con LLMs.",
             metadata={"topic": "tools"}),
    Document(page_content="Hugging Face es la plataforma líder para modelos de ML.",
             metadata={"topic": "tools"}),
]

# Add to vector store
loaded_vs.add_documents(new_docs)

print(f"Vectores después de añadir: {loaded_vs.index.ntotal}")

Vectores después de añadir: 12


<a name="ejercicios"></a>
## 7. Ejercicios Prácticos

### Ejercicio 1: Crear un buscador de FAQs

In [18]:
# Exercise 1: FAQ Search System

faqs = [
    {"q": "¿Cómo puedo resetear mi contraseña?", 
     "a": "Ve a Olvidé mi contraseña en la página de login."},
    {"q": "¿Cuál es el horario de atención?",
     "a": "Lunes a Viernes de 9:00 a 18:00."},
    {"q": "¿Aceptan devoluciones?",
     "a": "Sí, tienes 30 días para devolver productos sin usar."},
    {"q": "¿Tienen envío internacional?",
     "a": "Sí, enviamos a más de 50 países."},
     {"q": "¿Envian a Latinoamerica?",
     "a": "Enviamos a Argentina y Brasil"},
    {"q": "¿Cómo contacto con soporte?",
     "a": "Email: soporte@ejemplo.com o chat en vivo."},
]

# Create documents with questions and answers
faq_docs = [
   Document(
        page_content=f"Pregunta: {faq['q']} \nRespuesta: {faq['a']}",
        metadata={"pregunta_original": faq["q"], "answer": faq["a"]}
    )
    for faq in faqs
]

# Create vector store
faq_vs = FAISS.from_documents(faq_docs, embeddings)

# Search function
# Función de búsqueda mejorada con validación de distancia
def find_answer_improved(query, max_distance=0.8): 
    # max_distance dependerá del modelo de embeddings que uses (ej. OpenAI, HuggingFace)
    # Tendrás que hacer pruebas para encontrar tu número ideal.
    
    results = faq_vs.similarity_search_with_score(query, k=1)
    
    if results:
        doc, distance = results[0]
        print(f"[Debug] Distancia obtenida: {distance:.4f}") # Útil para calibrar tu umbral
        
        if distance <= max_distance:
            return doc.metadata["answer"]
            
    return "Lo siento, no encontré una respuesta exacta para tu consulta. ¿Puedes reformularla?"

# Test
test_queries = [
    "olvidé mi clave",
    "quiero devolver algo",
    "¿envían a México?"
]

print("Sistema FAQ:")
for q in test_queries:
    a = find_answer_improved(q)
    print(f"Q: {q}")
    print(f"A: {a}\n")

Sistema FAQ:
[Debug] Distancia obtenida: 1.2183
Q: olvidé mi clave
A: Lo siento, no encontré una respuesta exacta para tu consulta. ¿Puedes reformularla?

[Debug] Distancia obtenida: 0.9333
Q: quiero devolver algo
A: Lo siento, no encontré una respuesta exacta para tu consulta. ¿Puedes reformularla?

[Debug] Distancia obtenida: 0.6128
Q: ¿envían a México?
A: Enviamos a Argentina y Brasil



In [19]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

faqs = [
    {"q": "¿Cómo puedo resetear mi contraseña?", 
     "a": "Ve a Olvidé mi contraseña en la página de login."},
    {"q": "¿Cuál es el horario de atención?",
     "a": "Lunes a Viernes de 9:00 a 18:00."},
    {"q": "¿Aceptan devoluciones?",
     "a": "Sí, tienes 30 días para devolver productos sin usar."},
    {"q": "¿Tienen envío internacional?",
     "a": "Sí, enviamos a más de 50 países."},
     {"q": "¿Envian a Latinoamerica?",
     "a": "Enviamos a Argentina y Brasil"},
    {"q": "¿Cómo contacto con soporte?",
     "a": "Email: soporte@ejemplo.com o chat en vivo."},
]

# Create documents with questions and answers
faq_docs = [
    Document(
        page_content=f"Pregunta: {faq['q']} \nRespuesta: {faq['a']}",
        metadata={"pregunta_original": faq["q"], "answer": faq["a"]}
    )
    for faq in faqs
]

# Create vector store (asumo que 'embeddings' ya está definido arriba en tu entorno)
faq_vs = FAISS.from_documents(faq_docs, embeddings)

# Función de búsqueda modificada para retornar (respuesta, confianza_porcentaje, distancia_raw)
def find_answer_improved(query, max_distance=0.8): 
    results = faq_vs.similarity_search_with_score(query, k=1)
    
    if results:
        doc, distance = results[0]
        
        # Convertimos la distancia en un porcentaje estimado de confianza.
        # Nota: Si usas vectores normalizados (como OpenAI), la distancia L2 suele ir de 0 a 2.
        # Una fórmula matemática estándar para convertir distancia en confianza es: max(0, 1 - distancia)
        confianza = max(0.0, (1.0 - distance) * 100)
        
        if distance <= max_distance:
            return doc.metadata["answer"], confianza, distance
        else:
            # Si pasa del umbral, exponemos que no superó el filtro pero guardamos la métrica
            return "Lo siento, no encontré una respuesta exacta para tu consulta.", confianza, distance
            
    return "No se encontraron documentos en la base de datos.", 0.0, 2.0

# Test
test_queries = [
    "olvidé mi clave",          # Debería dar confianza alta
    "quiero devolver algo",     # Debería dar confianza alta
    "¿envían a México?",        # Debería dar confianza media/alta (por la palabra 'envían' y 'Latinoamérica')
    "¿qué tiempo hace hoy?"     # Debería dar confianza muy baja (y ser rechazado por el umbral)
]

print("--- Sistema FAQ con Nivel de Confianza --- \n")
for q in test_queries:
    ans, conf, dist = find_answer_improved(q)
    print(f"User Q: {q}")
    print(f"Bot  A: {ans}")
    print(f"📊 Confianza: {conf:.2f}% | Distancia FAISS: {dist:.4f}")
    
    # Añadimos una pequeña alerta visual si fue rechazado por el umbral
    if dist > 0.8:
        print("⚠️  [Nota: Esta respuesta se bloqueó o es genérica porque superó el umbral de distancia de 0.8]")
    print("-" * 50)

--- Sistema FAQ con Nivel de Confianza --- 

User Q: olvidé mi clave
Bot  A: Lo siento, no encontré una respuesta exacta para tu consulta.
📊 Confianza: 0.00% | Distancia FAISS: 1.2183
⚠️  [Nota: Esta respuesta se bloqueó o es genérica porque superó el umbral de distancia de 0.8]
--------------------------------------------------
User Q: quiero devolver algo
Bot  A: Lo siento, no encontré una respuesta exacta para tu consulta.
📊 Confianza: 6.67% | Distancia FAISS: 0.9333
⚠️  [Nota: Esta respuesta se bloqueó o es genérica porque superó el umbral de distancia de 0.8]
--------------------------------------------------
User Q: ¿envían a México?
Bot  A: Enviamos a Argentina y Brasil
📊 Confianza: 38.72% | Distancia FAISS: 0.6128
--------------------------------------------------
User Q: ¿qué tiempo hace hoy?
Bot  A: Lo siento, no encontré una respuesta exacta para tu consulta.
📊 Confianza: 11.39% | Distancia FAISS: 0.8861
⚠️  [Nota: Esta respuesta se bloqueó o es genérica porque superó el umb

### Ejercicio 2: Comparar diferentes chunk sizes

In [20]:
# Exercise 2: Experiment with chunk sizes
# Load a longer document and test different chunk sizes

# Create a longer sample document
long_document = """
===============================================================================
INFORME BIOGRÁFICO Y PROFESIONAL: JOAQUÍN DOMÍNGUEZ "EL XOKAS"
===============================================================================

1. INTRODUCCIÓN Y CONTEXTO
Joaquín Domínguez Portela, conocido internacionalmente en el mundo digital como
"El Xokas" o simplemente "Xokas", es uno de los creadores de contenido, streamers
y personalidades de internet más influyentes y polarizantes del mundo de habla
hispana. Nacido en Lugo, Galicia, España, el 13 de abril de 1997 (algunas fuentes
citan 1991), se ha consolidado como una de las figuras clave de plataformas de 
transmisión en vivo como Twitch y YouTube, destacando por su estilo directo, 
competitivo, y una personalidad sumamente histriónica y transparente que no deja 
indiferente a nadie.

A lo largo de su carrera, ha evolucionado desde ser un jugador experto y dedicado
de World of Warcraft (WoW) hasta convertirse en un entretenedor de masas, 
empresario del sector alimentario y analista de la actualidad de internet. Su
trayectoria combina récords históricos de audiencia con intensos debates mediáticos,
lo que lo convierte en un caso de estudio único sobre las dinámicas de la cultura
del streaming contemporánea.

===============================================================================
2. PRIMEROS AÑOS Y FORMACIÓN (VIDA ANTES DE INTERNET)
Antes de convertirse en una celebridad digital, Joaquín Domínguez llevó una vida
relativamente alejada del foco público masivo, aunque siempre vinculada a sus dos
grandes pasiones: los videojuegos y la comunicación.

Nacido y criado en Galicia, completó sus estudios universitarios licenciándose en
Comunicación Audiovisual, una disciplina que posteriormente admitiría que le otorgó 
herramientas técnicas y narrativas para entender el funcionamiento de los medios, 
la edición de vídeo y el ritmo del entretenimiento. Tras terminar la universidad, 
se mudó a Madrid, donde comenzó a trabajar en la industria tradicional como editor 
y postproductor de vídeo. Llegó a formar parte del equipo de edición del Real Madrid 
Club de Fútbol, una experiencia laboral exigente que moldeó su disciplina de trabajo, 
pero de la cual se terminó distanciando para apostar todo su tiempo y recursos a su 
canal de Twitch, un ecosistema que en aquel entonces (alrededor de 2017 y 2018) 
estaba empezando a explotar en España.

===============================================================================
3. TRAYECTORIA EN EL STREAMING Y CRONOLOGÍA DE SU ÉXITO
La carrera de El Xokas en el streaming se puede dividir en varias etapas críticas:

A. La etapa de World of Warcraft y el nicho (2018-2020)
Xokas comenzó en Twitch centrándose de manera casi obsesiva en World of Warcraft, 
específicamente en la vertiente JvJ (Jugador contra Jugador). Gracias a su profundo
conocimiento del juego y a su temperamento extremadamente competitivo y ruidoso, 
se convirtió rápidamente en el streamer de WoW más grande de España. Su comunidad, 
aunque pequeña en comparación con los gigantes del momento, era extremadamente 
fiel y activa. Durante este tiempo, acuñó frases y actitudes que definirían su marca: 
la meritocracia del esfuerzo, las sesiones maratónicas de juego y las respuestas 
directas y sin filtros a su chat.

B. La explosión de popularidad: "Egofand" y las series comunitarias (2021)
El verdadero punto de inflexión internacional llegó en 2021. La comunidad hispana
de Twitch comenzó a unirse en grandes eventos multijugador. El Xokas fue invitado a 
"Egoland" (un servidor de Rust que reunió a los mejores creadores). Su participación, 
caracterizada por un rol agresivo, competitivo y sumamente divertido, llamó la atención
de millones de espectadores que no lo conocían. Ese mismo año, su crecimiento fue 
exponencial, coronándose en los Premios ESLAND 2022 (celebrados en enero) como el 
"Streamer Revelación".

C. El hito histórico: Squid Craft Games (2022)
En enero de 2022, participó en los "Squid Craft Games", una serie en Minecraft basada 
en la popular serie de televisión "El juego del calamar". El Xokas llegó a la gran final 
en un infartante duelo contra el streamer OllieGamerz. El evento rompió récords globales: 
la transmisión de El Xokas alcanzó un pico de más de 1.2 millones de espectadores en 
simultáneo en su canal de Twitch, metiéndose directo en el top de las emisiones más 
vistas de la historia de la plataforma a nivel mundial.

D. Diversificación y madurez en el contenido (2023-Presente)
Tras tocar el cielo en audiencias, el contenido de Xokas mutó. Redujo las horas dedicadas
exclusivamente a videojuegos de nicho para centrarse en el "Just Chatting" (Charlas en 
general), reaccionando a la actualidad social, política y deportiva, debatiendo con 
su audiencia, realizando retos de cambio físico y alimentación, y expandiendo su huella
hacia plataformas como YouTube de manera más estructurada.

===============================================================================
4. CAMPOS DE EXPERTIZ Y CONTENIDOS DESTACADOS
El Xokas no es solo un jugador de videojuegos; su éxito radica en la especialización 
y el dominio de varias áreas específicas del entretenimiento digital:

- Videojuegos de Rol y Estrategia Competitiva (MMORPGs): Es una autoridad indiscutible 
  en World of Warcraft dentro de la comunidad hispanohablante. Sabe analizar dinámicas 
  de juego, parches, economía interna de los juegos y mecánicas competitivas a nivel profesional.
- Comunicación y Oratoria en Streaming: Su formación en Comunicación Audiovisual se nota 
  en su manejo del ritmo en pantalla. Es experto en mantener la atención de decenas de 
  miles de personas durante transmisiones de 8 a 12 horas consecutivas sin guion, utilizando 
  la modulación de voz, la narrativa improvisada y la interacción constante.
- Gestión de Comunidades de Internet (Engagement): Ha dominado el "fenómeno clip". 
  Sabe perfectamente qué declaraciones generarán impacto en redes sociales como TikTok y 
  X (Twitter), lo que le permite mantenerse en el ciclo de noticias de internet constantemente.
- El Sector "Fitness" y Nutrición (Etapa Reciente): En los últimos años, documentó de 
  manera abierta su proceso de cambio físico, transformando sus hábitos de vida y 
  convirtiendo la salud y el entrenamiento de fuerza en un pilar clave de su contenido, 
  colaborando con entrenadores de renombre.

===============================================================================
5. FACETA EMPRESARIAL: KNOWEATS!
Fiel a su discurso de evolución personal y de inversión del capital generado en internet, 
El Xokas dio un salto definitivo al mundo empresarial real en 2023. Adquirió, junto a su 
socio y también streamer Illojuan, la empresa de comida saludable a domicilio "Knoweats!".
Lejos de ser una simple campaña publicitaria, Joaquín se involucró de manera directa en 
la estrategia de negocio, la mejora de la infraestructura de producción en Villena 
(Alicante) y la promoción orgánica a través de sus canales. Esto le ha permitido destacar 
también como un caso de éxito en la reconversión de creadores de contenido en empresarios 
del sector tradicional (B2C).

===============================================================================
6. FILOSOFÍA DE VIDA, ESTILO Y CONTROVERSIAS
Es imposible analizar la figura de El Xokas sin abordar su fuerte carácter y las 
polémicas que lo han rodeado. Su lema implícito siempre ha sido la autenticidad radical: 
dice lo que piensa, a menudo con un lenguaje explícito, rudo y carente de filtros corporativos.

- La cultura del esfuerzo: Defiende férreamente que el éxito es el resultado del trabajo 
  duro y la obsesión sana por ser el mejor en tu disciplina. Desprecia la mediocridad y 
  suele utilizar discursos motivacionales muy agresivos orientados a que sus seguidores 
  tomen las riendas de su vida.
- Controversias mediáticas: Su estilo directo le ha costado múltiples crisis de reputación. 
  La más notoria ocurrió a principios de 2022, cuando se descubrió que utilizaba una cuenta 
  secundaria anónima en Twitter ("Cathuser8") para responder a sus críticos y defenderse 
  a sí mismo. Este evento abrió un debate internacional sobre la salud mental de los 
  creadores de contenido de élite y la presión a la que están sometidos. Xokas asumió el 
  error públicamente, se apartó temporalmente de las redes para reflexionar y regresó 
  reestructurando ciertos aspectos de su vida pública.

===============================================================================
7. IMPACTO CULTURAL Y LEGADO
Joaquín Domínguez "El Xokas" representa la viva imagen del streamer que no busca agradar 
a todo el mundo, sino construir una audiencia masiva basada en la lealtad inquebrantable 
de quienes aceptan su personalidad tal y como es. Ha demostrado que el carisma y la 
capacidad de debate pueden sostener un imperio digital incluso por encima de los propios 
videojuegos. Su legado técnico radica en haber elevado el listón de la retención de 
audiencia en el formato de "Just Chatting" y en servir de puente entre el entretenimiento 
digital de nicho y el ecosistema empresarial tradicional de gran escala.
===============================================================================

"""

# Test different chunk sizes and see search quality si es homogene el dara set chunk pequeño si no mas grande
chunk_sizes = [20, 100, 200, 400, 1000]
query = "Quien es el xokas"

print(f"Query: '{query}'\n")

for size in chunk_sizes:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=10)
    chunks = splitter.create_documents([long_document])
    vs_test = FAISS.from_documents(chunks, embeddings)
    results = vs_test.similarity_search(query, k=1)
    
    print(f"Chunk size={size}: {len(chunks)} chunks")
    print(f"  Mejor resultado: '{results[0].page_content[:200]}...'\n")

Query: 'Quien es el xokas'

Chunk size=20: 765 chunks
  Mejor resultado: 'de El Xokas en el...'

Chunk size=100: 120 chunks
  Mejor resultado: '"El Xokas" o simplemente "Xokas", es uno de los creadores de contenido, streamers...'

Chunk size=200: 63 chunks
  Mejor resultado: '1. INTRODUCCIÓN Y CONTEXTO
Joaquín Domínguez Portela, conocido internacionalmente en el mundo digital como
"El Xokas" o simplemente "Xokas", es uno de los creadores de contenido, streamers...'

Chunk size=400: 32 chunks
  Mejor resultado: '1. INTRODUCCIÓN Y CONTEXTO
Joaquín Domínguez Portela, conocido internacionalmente en el mundo digital como
"El Xokas" o simplemente "Xokas", es uno de los creadores de contenido, streamers
y personali...'

Chunk size=1000: 13 chunks
  Mejor resultado: '===============================================================================
INFORME BIOGRÁFICO Y PROFESIONAL: JOAQUÍN DOMÍNGUEZ "EL XOKAS"
=========================================================...'



## Resumen

En este notebook hemos aprendido:

1. **Vector Stores**: Bases de datos para búsqueda semántica
2. **FAISS**: Vector store local, rápido y gratuito
3. **Document Loaders**: Cargar desde archivos, web, PDFs
4. **Text Splitters**: Dividir documentos en chunks
5. **Similarity Search**: Buscar por significado
6. **Persistencia**: Guardar y cargar índices

### Arquitectura típica de un sistema de retrieval

```
Documentos → [Loader] → [Splitter] → [Embeddings] → [Vector Store]
                                                         ↓
Query → [Embedding] → [Similarity Search] → Documentos relevantes
```

En el siguiente notebook veremos cómo combinar esto con LLMs para crear sistemas **RAG** (Retrieval-Augmented Generation).

---

## Referencias

- [FAISS Documentation](https://faiss.ai/)
- [LangChain Vector Stores](https://python.langchain.com/docs/modules/data_connection/vectorstores/)
- [Chunking Strategies](https://www.pinecone.io/learn/chunking-strategies/)

In [21]:
import session_info
session_info.show(html = False)

-----
langchain_community         0.4.1
langchain_core              1.4.0
langchain_huggingface       NA
langchain_text_splitters    NA
session_info                v1.0.1
-----
IPython             9.1.0
jupyter_client      8.8.0
jupyter_core        5.9.1
jupyterlab          4.5.6
notebook            7.5.5
-----
Python 3.12.9 | packaged by Anaconda, Inc. | (main, Feb  6 2025, 13:04:33) [Clang 14.0.6 ]
macOS-10.16-x86_64-i386-64bit
-----
Session information updated at 2026-05-25 17:33
